# Phase 5 - Connectivity Engine Baseline Mapping

This notebook loads `baseline_connectivity_scores.csv` and visualizes tract-level accessibility under **Dry / Normal** and **Heavy Rain** scenarios.

In [ ]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

ROOT = Path('..').resolve()
scores_path = ROOT / 'data/processed/baseline_connectivity_scores.csv'
tracts_path = ROOT / 'data/processed/acs_demographics_centroids.geojson'

scores = pd.read_csv(scores_path, dtype={'tract_id': 'string'})
tracts = gpd.read_file(tracts_path).copy()
tracts['tract_id'] = tracts['geoid'].astype('string')

score_wide = scores.pivot(index='tract_id', columns='weather_scenario', values='connectivity_score').reset_index()
map_df = tracts.merge(score_wide, on='tract_id', how='left')

required_cols = ['Dry / Normal', 'Heavy Rain']
missing = [c for c in required_cols if c not in map_df.columns]
if missing:
    raise ValueError(f'Missing scenarios in baseline scores: {missing}')

denominator = map_df['Dry / Normal'].replace(0, pd.NA).astype('Float64')
map_df['heavy_vs_dry_pct'] = ((map_df['Heavy Rain'] - map_df['Dry / Normal']) / denominator * 100.0).astype('Float64')
map_df['heavy_vs_dry_pct'] = pd.to_numeric(map_df['heavy_vs_dry_pct'], errors='coerce')
map_df = map_df.to_crs('EPSG:4326')

print(f'Tracts loaded: {len(map_df)}')
map_df[['tract_id', 'Dry / Normal', 'Heavy Rain', 'heavy_vs_dry_pct']].head()

In [ ]:
sns.set_theme(style='white')
fig, axes = plt.subplots(1, 2, figsize=(15, 7), constrained_layout=True)

map_df.plot(
    column='Dry / Normal',
    cmap='YlGnBu',
    linewidth=0.1,
    edgecolor='white',
    legend=True,
    ax=axes[0],
    missing_kwds={'color': 'lightgrey'}
)
axes[0].set_title('Baseline Connectivity - Dry / Normal')
axes[0].axis('off')

map_df.plot(
    column='Heavy Rain',
    cmap='YlGnBu',
    linewidth=0.1,
    edgecolor='white',
    legend=True,
    ax=axes[1],
    missing_kwds={'color': 'lightgrey'}
)
axes[1].set_title('Baseline Connectivity - Heavy Rain')
axes[1].axis('off')

plt.show()

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 7))
map_df.plot(
    column='heavy_vs_dry_pct',
    cmap='RdBu',
    vmin=-100,
    vmax=20,
    linewidth=0.1,
    edgecolor='white',
    legend=True,
    ax=ax,
    missing_kwds={'color': 'lightgrey'},
    legend_kwds={'label': 'Percent change (%)'}
)
ax.set_title('Percent Change: Heavy Rain vs Dry / Normal')
ax.axis('off')
plt.tight_layout()
plt.show()

summary = scores.groupby('weather_scenario')['connectivity_score'].agg(['mean', 'median', 'min', 'max']).sort_values('mean', ascending=False)
summary

In [ ]:
impact = map_df[['tract_id', 'Dry / Normal', 'Heavy Rain', 'heavy_vs_dry_pct']].copy()
impact = impact.dropna(subset=['heavy_vs_dry_pct']).sort_values('heavy_vs_dry_pct')
impact.head(15)